# Новости → «бак» Орнштейна–Уленбека → доходность «закрытие → закрытие»

Эта тетрадка — вторая постановка курсовой, **ближе всего к буквальному плану**:

$$
x_{t+1}=A\,x_t+\varepsilon_t,\qquad p_t=\langle g,x_t\rangle .
$$

Здесь $p_t$ — логарифм цены **закрытия** дня $t$, а $\varepsilon_t$ — новости торгового дня $t$, то есть всё, что вышло
между закрытием $t-1$ и закрытием $t$ (окно `trading_date` из датасета). Если вычесть $p_{t-1}$ из $p_t$, получится
доходность за день:

$$
r_t = p_t-p_{t-1}=\langle g,\;x_t-x_{t-1}\rangle=\langle g,\;(A-I)\,x_{t-1}+\varepsilon_{t-1}\rangle .
$$

Обратите внимание на индексы: в правой части только $\varepsilon_{t-1}$ и раньше. Значит, в индексации плана модель
**предсказывает** доходность дня $t$ по новостям, известным к закрытию $t-1$. Это честный прогноз: решение можно
принять на закрытии $t-1$ (заявка «по цене закрытия») и держать позицию до закрытия $t$.

```
 закрытие t−2 ──────── новости ε_{t−1} ──────── закрытие t−1 ─────── доходность r_t ─────── закрытие t
```

### Что в этой тетрадке

| № | раздел | что делаем |
|---|---|---|
| 1 | данные | окна новостей `trading_date`, доходности «закрытие → закрытие» и их части (ночь и сессия) |
| 2 | эмбеддинги $\varepsilon_t$ | те же, что в первой тетрадке: объём, словарь, TF-IDF, SBERT (кэш общий) |
| 3 | модель | две формы: **буквальная** (разность состояний) и **уровень** (сами состояния) |
| 4 | обучение | ridge-регрессия + подбор памяти по сетке; проверка на синтетике |
| 5 | прогноз знака и величины доходности | OOS $R^2$, Кларк–Уэст, доля угаданных знаков, накопленная ошибка, простая стратегия, по годам |
| 6 | прогноз модуля доходности | волатильность «закрытие → закрытие» против HAR + VIX |
| 7 | объяснение того же дня | новости дня $t$ → доходность дня $t$; профиль по сдвигу и фильтр рыночных сводок |
| 8 | ночь и сессия | где именно сидит связь новостей с ценой |
| 9 | итоги | |

### Чего ждать заранее

Знак дневной доходности индекса — одна из самых трудно предсказуемых величин в финансах: если бы новости газеты
предсказывали его заметно, этим давно бы пользовались. Типичный честный результат в литературе — OOS $R^2$ порядка
десятых долей процента или около нуля. Поэтому главный вопрос тетрадки — **есть ли хоть какой-то устойчивый сигнал**
и как он устроен: мгновенная реакция, откат (избыточная реакция) или продолжение (недостаточная реакция).

### Словарик

| термин | простыми словами |
|---|---|
| **доходность «закрытие → закрытие»** | $r_t=100\cdot\ln(C_t/C_{t-1})$, в процентах; $C$ — цена закрытия с поправкой на дивиденды |
| **ночной гэп и сессия** | $r_t$ = гэп (закрытие $t-1$ → открытие $t$) + сессия (открытие $t$ → закрытие $t$) |
| **OOS $R^2$** | доля разброса, объяснённая лучше, чем прогнозом «историческое среднее»; > 0 — модель полезна |
| **тест Кларка–Уэста** | значимо ли модель лучше исторического среднего (для вложенных моделей) |
| **доля угаданных знаков** | как часто знак прогноза совпадает со знаком доходности; сравнивается не с 50%, а с тестом Песарана–Тиммерманна |
| **накопленная разница ошибок** | график, на котором видно, *когда* модель выигрывала у среднего |
| **ridge** | линейная регрессия со штрафом за большие коэффициенты |
| **период полураспада $h$** | через сколько календарных дней влияние новости уменьшается вдвое |

In [ ]:
import hashlib
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import lfilter
from scipy.stats import norm, spearmanr
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from scripts import config as C
from scripts.align import session_open_utc
from scripts.news import article_text

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "figure.figsize": (11, 3.6)})

# ---------- что и на каком периоде (как в первой тетрадке: кэш SBERT общий) ----------
TICKER = "SPY"
START, END = "2008-01-01", "2023-04-30"
TRAIN_END, VAL_END = "2016-12-31", "2019-12-31"
# ---------- модель ----------
HALF_LIVES = [0, 1, 2, 3, 5, 7, 14, 30, 60]
ALPHAS = [10, 100, 1000, 10000]      # для знака доходности сигнал очень слабый — нужны и сильные штрафы
D_EMB = 8
Z_WINDOW = 252
WARMUP = 252
USE_SBERT = True
SBERT_DEVICE = None                  # None — автоматически: cuda / mps / cpu
SBERT_BATCH = 256
# ---------- проверки ----------
COST_BP = 1.0                        # издержки стратегии: базисных пунктов за единицу оборота
SEED = 0

def make_rng(*keys):
    return np.random.default_rng([SEED, *keys])

---
# 1. Данные

Берём тот же датасет и тот же период, что в первой тетрадке (2008 → апрель 2023: есть время публикации и настоящие
рубрики). Новости — финансовые статьи. В отличие от первой тетрадки, здесь в окно дня $t$ входят **все** статьи этого
торгового дня — от закрытия $t-1$ до закрытия $t$, — потому что и цель считается по закрытиям.

In [ ]:
prices = pd.read_parquet(C.PROCESSED_FILES["prices"])
prices["date"] = pd.to_datetime(prices["date"])

NEWS_COLS = ["pub_time_utc", "trading_date", "is_finance", "flag_source", "headline", "abstract", "lead_paragraph"]
try:
    news = pd.read_parquet(C.PROCESSED_FILES["news"], columns=NEWS_COLS,
                           filters=[("trading_date", ">=", pd.Timestamp(START))])
except Exception:
    news = pd.read_parquet(C.PROCESSED_FILES["news"], columns=NEWS_COLS)
news["trading_date"] = pd.to_datetime(news["trading_date"])
news = news[news["trading_date"] >= START]
end = pd.Timestamp(END)
guessed = news.loc[news["flag_source"] == "text_model", "trading_date"]
if len(guessed):
    end = min(end, guessed.min() - pd.Timedelta(days=1))

def ohlc(ticker):
    p = prices[prices["ticker"] == ticker].set_index("date").sort_index()
    f = p["adj_close"] / p["close"]
    return pd.DataFrame({"open": p["open"] * f, "high": p["high"] * f, "low": p["low"] * f, "close": p["adj_close"]})

full = ohlc(TICKER)
dates = full.loc[START:end].index
gap_days = dates.to_series().diff().dt.days.fillna(1).to_numpy()
masks = {"train": np.asarray(dates <= TRAIN_END),
         "val": np.asarray((dates > TRAIN_END) & (dates <= VAL_END)),
         "test": np.asarray(dates > VAL_END)}

# доходности в процентах; всё на полной истории, потом режем по датам
full["r"] = 100 * np.log(full["close"]).diff()                       # закрытие → закрытие
full["gap"] = 100 * np.log(full["open"] / full["close"].shift(1))    # ночь: закрытие t−1 → открытие t
full["sess"] = 100 * np.log(full["close"] / full["open"])            # сессия: открытие → закрытие
full["abs_y"] = np.log(full["r"].abs() + 0.01)                       # log|r| — мера волатильности
M = full.reindex(dates)
r = M["r"].to_numpy()

vix = prices[prices["ticker"] == "^VIX"].set_index("date")["close"].sort_index()
M["log_vix_lag"] = np.log(vix).shift(1).reindex(dates)              # VIX на закрытии t−1

for k, m in masks.items():
    print(f"{k:5s} {dates[m][0].date()} → {dates[m][-1].date()}  {m.sum()} дней")
print(f"r_t: среднее {np.nanmean(r):+.3f}%, ст. откл. {np.nanstd(r):.2f}%, доля дней роста {np.nanmean(r > 0):.1%}")
print(f"проверка: r = гэп + сессия, макс. расхождение {np.nanmax(np.abs(M['r'] - M['gap'] - M['sess'])):.2e}")

### 1.1. Новости дня и из чего состоит окно

Окно торгового дня $t$ = (закрытие $t-1$, закрытие $t$]. Внутри него три части: вечер после закрытия, ночь и утро до
открытия, сама сессия. Отметим статьи, вышедшие до открытия, — они понадобятся в разделе 8.

In [ ]:
open_utc = pd.Series(session_open_utc(dates).tz_convert("UTC").tz_localize(None), index=dates)
news = news[news["is_finance"] & news["trading_date"].isin(dates)].copy()
pub = news["pub_time_utc"].dt.tz_convert("UTC").dt.tz_localize(None)
news["pre_open"] = (pub <= news["trading_date"].map(open_utc)).to_numpy()

# две группы — в том же порядке, что в первой тетрадке, чтобы векторы SBERT взялись из общего кэша
fin_pre = news[news["pre_open"]]
fin_rest = news[~news["pre_open"]]

hour = news["pub_time_utc"].dt.tz_convert("America/New_York")
hh = hour.dt.hour + hour.dt.minute / 60
same_day = (hour.dt.tz_localize(None).dt.normalize() == news["trading_date"]).to_numpy()
part = np.where(~same_day, "вечер t−1 и выходные", np.where(hh <= 9.5, "ночь и утро", "сессия t"))
display((pd.Series(part).value_counts(normalize=True) * 100).round(1).to_frame("% статей окна"))
print(f"финансовых статей: {len(news):,}, в среднем {len(news) / len(dates):.1f} в торговый день")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
lags = range(1, 21)
ax[0].bar(list(lags), [pd.Series(r)[np.isfinite(r)].autocorr(l) for l in lags])
ax[0].set_title("автокорреляция r_t: почти ноль — знак плохо предсказуем по прошлому")
ax[1].plot(dates, M["r"].rolling(63).std(), lw=1)
ax[1].set_title("волатильность r_t (скользящее ст. откл., 63 дня): меняется сильно")
plt.tight_layout(); plt.show()

**Что видно.** Автокорреляция доходности около нуля — по прошлым доходностям знак почти не предсказать. Волатильность,
наоборот, меняется сильно и медленно. Поэтому ниже прогнозируем и сам $r_t$ (раздел 5), и его модуль (раздел 6).

**Итог раздела 1:** для каждого торгового дня есть доходность и её части, а также все финансовые статьи окна этого дня.

---
# 2. Эмбеддинги новостей $\varepsilon_t$

Те же четыре способа, что в первой тетрадке, только по **всем статьям окна** дня $t$:

| название | что считаем |
|---|---|
| `volume` | log(1 + число статей / Δt) |
| `lexicon` | доли негативных, позитивных и «неопределённых» слов на 1000 слов — для знака доходности это главный кандидат |
| `tfidf` | TF-IDF + SVD до 8 чисел (обучение на train) |
| `sbert` | вектор каждой статьи, среднее за день, PCA до 8 чисел (обучение на train) |

Каждый признак очищаем скользящим z-score по прошлому году. Функция `build_eps` принимает маски статей — так в
разделах 7–8 те же эмбеддинги строятся по части статей (без рыночных сводок, только утро, только сессия).

In [ ]:
tr = masks["train"]

NEG = """loss losses lose losing fall falls fell falling drop drops dropped decline declines declined plunge plunged
slump slumped crash crisis fear fears worry worries worried recession default defaults bankruptcy bankrupt layoffs cuts weak
weakness slowdown downturn selloff turmoil panic collapse fail failed failure threat threatens trouble troubled fraud
debt deficit unemployment""".split()
POS = """gain gains gained rise rises rose rising rally rallied surge surged jump jumped growth grow grew strong strength
recovery recover rebound boost improve improved improvement profit profits profitable optimism optimistic upbeat beat""".split()
UNC = """uncertain uncertainty unclear volatile volatility may might could possibly unsettled doubt doubts concern concerns
concerned unpredictable turbulence risk risks risky""".split()

def lexicon_features(texts):
    vocab = sorted(set(NEG + POS + UNC))
    col = {w: i for i, w in enumerate(vocab)}
    cnt = CountVectorizer(vocabulary=vocab, token_pattern=r"(?u)\b\w+\b").transform(texts)
    n_words = texts.str.split().str.len().fillna(0).clip(lower=1).to_numpy()
    rate = lambda ws: 1000 * np.asarray(cnt[:, [col[w] for w in set(ws)]].sum(axis=1)).ravel() / n_words
    return pd.DataFrame({"neg_rate": rate(NEG), "pos_rate": rate(POS), "unc_rate": rate(UNC)}, index=texts.index)

def tfidf_features(texts, d=D_EMB):
    vec = TfidfVectorizer(min_df=5, max_df=0.5, max_features=30_000, ngram_range=(1, 2),
                          sublinear_tf=True, stop_words="english").fit(texts[tr])
    Xs = vec.transform(texts)
    svd = TruncatedSVD(n_components=d, algorithm="arpack", random_state=SEED).fit(Xs[tr])
    return pd.DataFrame(svd.transform(Xs), index=dates, columns=[f"svd{i + 1}" for i in range(d)])

def past_zscore(df, window=Z_WINDOW, minp=60, clip=5):
    mu = df.rolling(window, min_periods=minp).mean().shift(1)
    sd = df.rolling(window, min_periods=minp).std().shift(1)
    return ((df - mu) / sd).clip(-clip, clip).fillna(0.0)

# ---------- SBERT: тот же кэш, что в первой тетрадке ----------
SBERT_MODEL = None
def sbert_device():
    if SBERT_DEVICE:
        return SBERT_DEVICE
    try:
        import torch
    except ImportError:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    return "mps" if mps is not None and mps.is_available() else "cpu"

def sbert_encode(arts):
    global SBERT_MODEL
    key = hashlib.md5("\n".join(arts).encode("utf-8")).hexdigest()[:12]
    cache = C.EMBED_DIR / f"preopen_minilm_{key}.npz"
    if cache.exists():
        return np.load(cache)["emb"]
    from sentence_transformers import SentenceTransformer
    dev = sbert_device()
    if SBERT_MODEL is None:
        SBERT_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev)
    print(f"SBERT: кодируем {len(arts):,} статей на {dev}")
    emb = np.asarray(SBERT_MODEL.encode(list(arts), batch_size=SBERT_BATCH if dev != "cpu" else 64,
                                        show_progress_bar=True, normalize_embeddings=True), dtype=np.float32)
    C.EMBED_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(cache.stem + ".tmp.npz")
    np.savez_compressed(tmp, emb=emb)
    tmp.replace(cache)
    return emb

GROUPS = {"pre": fin_pre, "rest": fin_rest}
SB_EMB = {}
if USE_SBERT:
    try:
        for g, df in GROUPS.items():
            SB_EMB[g] = sbert_encode(article_text(df).to_numpy())
    except ImportError:
        print("sentence-transformers не установлен — SBERT пропускаем")
        SB_EMB = {}

def build_eps(masks_by_group=None, kinds=("volume", "lexicon", "tfidf", "sbert")):
    # ε_t по статьям окна trading_date; masks_by_group — какие статьи брать в каждой группе (по умолчанию все)
    masks_by_group = masks_by_group or {}
    sel = {g: (masks_by_group.get(g, np.ones(len(df), bool))) for g, df in GROUPS.items()}
    dfs = [GROUPS[g][sel[g]] for g in GROUPS]
    allart = pd.concat(dfs).sort_index(kind="stable")
    texts = article_text(allart).groupby(allart["trading_date"].to_numpy()).agg("\n".join).reindex(dates).fillna("")
    n = allart.groupby("trading_date").size().reindex(dates).fillna(0).to_numpy()
    raw = {}
    if "volume" in kinds:
        raw["volume"] = pd.DataFrame({"log_n": np.log1p(n / gap_days)}, index=dates)
    if "lexicon" in kinds:
        raw["lexicon"] = lexicon_features(texts)
    if "tfidf" in kinds:
        raw["tfidf"] = tfidf_features(texts)
    if "sbert" in kinds and SB_EMB:
        S, cnt = 0, 0
        for g, df in GROUPS.items():
            key = df["trading_date"].to_numpy()[sel[g]]
            S = S + pd.DataFrame(SB_EMB[g][sel[g]]).groupby(key).sum().reindex(dates).fillna(0.0)
            cnt = cnt + pd.Series(key).value_counts().reindex(dates).fillna(0)
        day = S.div(cnt.replace(0, np.nan), axis=0)
        day = day.fillna(day.loc[tr].mean())
        pca = PCA(n_components=D_EMB, svd_solver="full").fit(day.loc[tr])
        raw["sbert"] = pd.DataFrame(pca.transform(day), index=dates, columns=[f"sbert{i + 1}" for i in range(D_EMB)])
    return {k: past_zscore(v) for k, v in raw.items()}

EMB = build_eps()
for k, v in EMB.items():
    print(f"ε_t [{k:8s}]: {v.shape[1]} чис. в день")

**Итог раздела 2:** для каждого торгового дня есть $\varepsilon_t$ по всем финансовым статьям окна `trading_date = t`.
Важно: $\varepsilon_t$ известен только **после закрытия** дня $t$, поэтому для прогноза $r_t$ его использовать нельзя —
только $\varepsilon_{t-1}$ и раньше.

---
# 3. Модель: бак и две формы связи с ценой

**Бак.** Как в первой тетрадке: $S_t=A^{\Delta t}S_{t-1}+\varepsilon_t$, $A=\mathrm{diag}(0.5^{1/h})$ — состояние после
новостей дня $t$ (на закрытии $t$). В обозначениях плана $x_t=S_{t-1}$: состояние, известное к закрытию $t-1$.

**Две формы прогноза $r_t$** (обе используют только информацию до закрытия $t-1$):

| форма | формула | смысл |
|---|---|---|
| **буквальная** (разность) | $r_t\approx g_0+\langle g,\,x_t-x_{t-1}\rangle=g_0+\langle g,\,(A^{\Delta t}-I)x_{t-1}+\varepsilon_{t-1}\rangle$ | ровно план $p_t=\langle g,x_t\rangle$: цена — это «прочитанный» бак, доходность — изменение бака |
| **уровень** | $r_t\approx g_0+\langle g,\,x_t\rangle$ | доходность пропорциональна накопленному «настроению» новостей — классическая предсказательная регрессия |

В буквальной форме видна интересная вещь: вклад вчерашних новостей $\varepsilon_{t-1}$ входит с коэффициентом $g$, а
накопленный бак — с коэффициентом $(A-I)g$, то есть **с противоположным знаком** (потому что $A$ сжимающая). Если эта
форма подходит данным, рынок реагирует на новость, а потом частично «откатывает» реакцию: цена следует за баком,
а бак затухает.

Бак считается быстро: на календарной сетке (в выходные нули) рекурсия — это стандартный фильтр `lfilter`.

In [ ]:
def ou_state(eps, gaps, half_life):
    # S_t = A^{Δt} S_{t-1} + ε_t, A = diag(0.5^(1/h)); быстро через календарную сетку и lfilter
    E = np.asarray(eps, dtype=float)
    hl = np.broadcast_to(np.asarray(half_life, dtype=float), (E.shape[1],))
    pos = np.r_[0, np.cumsum(np.round(np.asarray(gaps, dtype=float)[1:]).astype(int))]
    G = np.zeros((pos[-1] + 1, E.shape[1]))
    G[pos] = E
    X = E.copy()
    for h in np.unique(hl[hl > 0]):
        cols = hl == h
        X[:, cols] = lfilter([1.0], [1.0, -0.5 ** (1.0 / h)], G[:, cols], axis=0)[pos]
    return X

def shift_rows(X, k):
    # сдвиг вниз на k строк: строка t получает значение строки t−k (k < 0 — вверх, «из будущего»)
    X = np.asarray(X, dtype=float)
    out = np.full_like(X, np.nan)
    if k > 0:
        out[k:] = X[:-k]
    elif k < 0:
        out[:k] = X[-k:]
    else:
        out = X.copy()
    return out

def news_design(E, h, mode, lag=1):
    # признаки для r_t: lag=1 — прогноз (новости до закрытия t−1), lag=0 — объяснение (новости дня t)
    S = ou_state(E, gap_days, h)
    F = S if mode == "уровень" else S - shift_rows(S, 1)
    return shift_rows(F, lag)

---
# 4. Обучение

**Веса $g$ — ridge-регрессия** точной формулой: признаки стандартизуем по обучающей части, свободный член — среднее
$y$, $\hat g=(Z^\top Z+\alpha I)^{-1}Z^\top(y-\bar y)$. Штраф $\alpha$ здесь особенно важен: сигнал в знаке доходности
крошечный, и без штрафа регрессия подгоняется под шум.

**Память $h$ и штраф $\alpha$ — перебор по сетке**: для каждой пары обучаем $g$ на train (2008–2016), считаем ошибку на
val (2017–2019), берём лучшую; финальная модель переобучается на train+val и один раз оценивается на тесте (2020 →).

**Бенчмарк для знака доходности — историческое среднее** (стандарт в литературе: Welch & Goyal, 2008; Campbell &
Thompson, 2008). Модель со всеми $g=0$ превращается ровно в него, поэтому для сравнения подходит тест Кларка–Уэста.

In [ ]:
def fit_predict(F, y, fit_mask, alpha):
    F, y = np.asarray(F, dtype=float), np.asarray(y, dtype=float)
    ok_ = np.isfinite(F).all(axis=1) & np.isfinite(y)
    m = fit_mask & ok_
    mu, sd = F[m].mean(axis=0), F[m].std(axis=0)
    sd = np.where(sd > 0, sd, 1.0)
    Z = (F[m] - mu) / sd
    y0 = y[m].mean()
    g = np.linalg.solve(Z.T @ Z + alpha * np.eye(Z.shape[1]), Z.T @ (y[m] - y0))
    pred = np.full(len(y), np.nan)
    pred[ok_] = y0 + ((F[ok_] - mu) / sd) @ g
    return pred, g

def mse(y, p, m):
    return np.mean((y[m] - p[m]) ** 2)

def select(make_F, y, okv, half_lives=HALF_LIVES, alphas=ALPHAS):
    trm, vam = masks["train"] & okv, masks["val"] & okv
    best = None
    for h in half_lives:
        F = make_F(h)
        fin = np.isfinite(F).all(axis=1)
        for a in alphas:
            p = fit_predict(F, y, trm & fin, a)[0]
            v = vam & fin
            s = np.mean((y[v] - p[v]) ** 2)
            if best is None or s < best[0]:
                best = (s, h, a)
    r2_val = 1 - best[0] / np.mean((y[vam] - y[trm].mean()) ** 2)
    return {"half_life": best[1], "alpha": best[2], "R² val": r2_val}

# ---------- метрики ----------
def oos_r2(y, p, m, ref):
    return 1 - np.sum((y[m] - p[m]) ** 2) / np.sum((y[m] - ref) ** 2)

def newey_west_se(x, lags=10):
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    s = x @ x / n
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * (x[l:] @ x[:-l]) / n
    return np.sqrt(max(s, 0) / n)

def clark_west(y, p_base, p_new, lags=10):
    f = (y - p_base) ** 2 - (y - p_new) ** 2 + (p_base - p_new) ** 2
    t = f.mean() / newey_west_se(f, lags)
    return t, 1 - norm.cdf(t)

def pesaran_timmermann(y, f):
    # доля угаданных знаков и односторонний тест «лучше, чем при независимости прогноза и факта»
    y, f = np.asarray(y) > 0, np.asarray(f) > 0
    n = len(y)
    py, px = y.mean(), f.mean()
    p_hat = (y == f).mean()
    p_star = py * px + (1 - py) * (1 - px)
    v_hat = p_star * (1 - p_star) / n
    v_star = ((2 * py - 1) ** 2 * px * (1 - px) + (2 * px - 1) ** 2 * py * (1 - py)) / n \
             + 4 * py * px * (1 - py) * (1 - px) / n ** 2
    d = v_hat - v_star
    if d <= 0:
        return p_hat, np.nan
    return p_hat, 1 - norm.cdf((p_hat - p_star) / np.sqrt(d))

def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj

### 4.1. Проверка на синтетике

Генерируем мир строго по плану: новости → бак с памятью **7 дней** → цена $p_t=\langle g,x_t\rangle+m_t$, где $m_t$ —
случайное блуждание (вся та часть цены, что не связана с нашими новостями). Доля доходности, объяснённая новостями,
— `share`. Затем обучаем **буквальную** форму и смотрим, находит ли перебор правильную память. Если при реалистично
слабом сигнале не находит — точное значение $h$ на рынке определить не удастся.

In [ ]:
def synthetic(share, seed, d=3, true_h=7):
    rg = make_rng(101, seed)
    E = rg.standard_normal((len(dates), d))
    x_plan = shift_rows(ou_state(E, gap_days, true_h), 1)           # x_t = S_{t−1}
    sig = np.nan_to_num(np.diff(np.r_[0.0, np.nan_to_num(x_plan @ rg.standard_normal(d))]))
    sig = (sig - sig.mean()) / sig.std()
    ys = np.sqrt(share) * sig + np.sqrt(1 - share) * rg.standard_normal(len(dates))   # r_t = сигнал + шум
    allok = np.ones(len(dates), bool)
    curve = []
    for h in HALF_LIVES:
        F = news_design(E, h, "разность", lag=1)
        fin = np.isfinite(F).all(axis=1)
        p = fit_predict(F, ys, masks["train"] & fin, 10)[0]
        v = masks["val"] & fin
        curve.append(oos_r2(ys, p, v, ys[masks["train"]].mean()))
    return np.array(curve)

fig, axs = plt.subplots(1, 3, figsize=(13, 3.3))
for ax, sh in zip(axs, [0.10, 0.02, 0.005]):
    runs = np.array([synthetic(sh, s) for s in range(15)])
    best_h = np.array(HALF_LIVES)[runs.argmax(axis=1)]
    xs = np.arange(len(HALF_LIVES))
    ax.plot(xs, runs.mean(0), marker="o"); ax.fill_between(xs, runs.mean(0) - runs.std(0), runs.mean(0) + runs.std(0), alpha=0.2)
    ax.axvline(HALF_LIVES.index(7), color="k", ls="--", lw=1)
    ax.set_xticks(xs); ax.set_xticklabels(HALF_LIVES)
    ax.set_title(f"новости = {sh:.1%} дисперсии r_t\nh ∈ [5, 14] в {np.isin(best_h, [5, 7, 14]).mean():.0%} запусков")
    ax.set_xlabel("h, дни"); ax.set_ylabel("OOS R² на val")
plt.tight_layout(); plt.show()

**Как читать.** При сильном сигнале пик — на истинных 7 днях: код и буквальная форма работают. При долях процента
(реалистично для дневной доходности) кривая почти плоская, и выбранный $h$ почти случаен. Это заранее говорит, чего
ждать от реальных данных: даже если сигнал есть, «скорость забывания» новостей по одной лишь доходности индекса
определяется плохо.

---
# 5. Прогноз доходности $r_t$

Для каждого эмбеддинга и каждой формы (буквальная / уровень) подбираем $h$ и $\alpha$ на валидации, переобучаем на
train+val и считаем на тесте:

* **OOS $R^2$** против исторического среднего (train+val);
* **CW p** — тест Кларка–Уэста против среднего, и он же с поправкой Холма по всей таблице;
* **знаки**: доля угаданных знаков прогноза *относительно среднего* (выше/ниже обычного дня) и p-value теста
  Песарана–Тиммерманна;
* **IC** — ранговая корреляция прогноза и факта.

In [ ]:
ok_r = np.isfinite(r) & (np.arange(len(dates)) >= WARMUP)
TRV, VAL, TST = masks["train"] & ok_r, masks["val"] & ok_r, masks["test"] & ok_r
TV = TRV | VAL
ref_r = r[TV].mean()

def evaluate_signed(E, mode, lag=1, y=r, okv=ok_r):
    s = select(lambda h: news_design(E, h, mode, lag), y, okv)
    F = news_design(E, s["half_life"], mode, lag)
    tv, te = (masks["train"] | masks["val"]) & okv, masks["test"] & okv
    fin = np.isfinite(F).all(axis=1)
    p = fit_predict(F, y, tv & fin, s["alpha"])[0]
    ref = y[tv].mean()
    te = te & fin
    base = np.full(len(y), ref)
    hit, pt_p = pesaran_timmermann(y[te] - ref, p[te] - ref)
    return {"h": s["half_life"], "α": s["alpha"], "R² val": s["R² val"],
            "OOS R² test, %": 100 * oos_r2(y, p, te, ref),
            "CW p": clark_west(y[te], base[te], p[te])[1],
            "знаки угаданы": hit, "PT p": pt_p,
            "IC test": spearmanr(p[te], y[te])[0] if np.std(p[te]) > 0 else np.nan}, p

rows, PRED = [], {}
for emb in EMB:
    for mode in ("разность", "уровень"):
        res, p = evaluate_signed(EMB[emb].to_numpy(), mode)
        rows.append({"эмбеддинг": emb, "форма": mode, **res})
        PRED[(emb, mode)] = p
signed = pd.DataFrame(rows)
signed["CW p (Холм)"] = holm(signed["CW p"])
display(signed.set_index(["эмбеддинг", "форма"]).round(4))

best_key = tuple(signed.loc[signed["R² val"].idxmax(), ["эмбеддинг", "форма"]])
print("лучшая по валидации модель:", best_key)

**Как читать.**

* **OOS $R^2$ > 0 и CW p (Холм) < 0.05** — новости предсказывают доходность лучше исторического среднего. Для
  дневной доходности индекса даже +0.5% — заметный результат.
* **Доля угаданных знаков** около доли дней роста (≈ 53–54%) — модель угадывает не лучше «всегда рост». Песаран–
  Тиммерманн корректно учитывает именно это.
* **Сравнение форм.** Если буквальная форма лучше уровня, данные согласуются с планом $p_t=\langle g,x_t\rangle$:
  реакция на новость с последующим откатом. Если лучше уровень — скорее «дрейф» в сторону накопленного настроения.
* У лучшей модели посмотрите на $h$: при плоской кривой (раздел 4.1) значение памяти мало что значит.

### 5.1. Когда модель выигрывала: накопленная разница ошибок

Для лучшей по валидации модели рисуем $\sum_s\left[(r_s-\bar r)^2-(r_s-\hat r_s)^2\right]$ по дням теста (и валидации для
сравнения). Рост линии — модель лучше среднего в эти дни, падение — хуже. Видно, набран ли результат равномерно или за
несколько дней (например, в марте 2020).

In [ ]:
p_best = PRED[best_key]
fig, ax = plt.subplots()
for name, m, c in (("val", VAL, "C1"), ("test", TST, "C2")):
    mm = m & np.isfinite(p_best)
    d = np.where(mm, (r - ref_r) ** 2 - (r - p_best) ** 2, 0.0)
    ax.plot(dates[mm], np.cumsum(d[mm]), color=c, label=name)
ax.axhline(0, color="k", lw=0.8)
ax.set_title(f"накопленная разница квадратов ошибок «среднее − модель», {best_key[0]} / {best_key[1]}")
ax.set_ylabel("%²"); ax.legend()
plt.tight_layout(); plt.show()

### 5.2. Есть ли экономический смысл: простая стратегия

Позиция на день $t$ решается на закрытии $t-1$ по прогнозу: **long/short** — покупаем, если прогноз выше исторического
среднего, иначе продаём; **long/cash** — покупаем, если выше среднего, иначе в деньгах. Издержки — `COST_BP` базисных
пунктов за каждую единицу оборота позиции. Сравниваем с «купил и держи» на тесте. Это иллюстрация, а не торговая
система: один инструмент, без учёта проскальзывания и налогов.

In [ ]:
def strategy(rp, w, cost_bp=COST_BP):
    R = np.expm1(rp / 100)
    turn = np.abs(np.diff(np.r_[0.0, w]))
    s = w * R - turn * cost_bp / 1e4
    eq = np.cumprod(1 + s)
    return {"доходность, % годовых": 100 * (eq[-1] ** (252 / len(s)) - 1),
            "волатильность, %": 100 * s.std() * np.sqrt(252),
            "Шарп": s.mean() / s.std() * np.sqrt(252) if s.std() > 0 else np.nan,
            "макс. просадка, %": 100 * (eq / np.maximum.accumulate(eq) - 1).min(),
            "оборот в год": turn.sum() * 252 / len(s)}, eq

te = TST & np.isfinite(p_best)
signal = p_best[te] - ref_r
strat = {"купил и держи": np.ones(te.sum()),
         "long/short по прогнозу": np.sign(signal),
         "long/cash по прогнозу": (signal > 0).astype(float)}
rows, curves = [], {}
for name, w in strat.items():
    st, eq = strategy(r[te], w)
    rows.append({"стратегия": name, **st}); curves[name] = eq
display(pd.DataFrame(rows).set_index("стратегия").round(3))

fig, ax = plt.subplots()
for name, eq in curves.items():
    ax.plot(dates[te], eq, label=name)
ax.set_title("тест: рост 1 доллара"); ax.legend()
plt.tight_layout(); plt.show()

**Как читать.** Шарп стратегии выше, чем у «купил и держи», и это сохраняется после издержек — у прогноза есть
экономический смысл. Если стратегия выигрывает только за счёт того, что сидела в деньгах в марте 2020, это один
эпизод, а не устойчивое преимущество (см. разбивку по годам ниже).

### 5.3. По годам

Для лучшей модели (настройки фиксированы) — расширяющееся окно: обучаем на всех данных до 1 января года $Y$,
предсказываем год $Y$ и считаем OOS $R^2$ против исторического среднего на том же окне обучения.

In [ ]:
emb_b, mode_b = best_key
row_b = signed.set_index(["эмбеддинг", "форма"]).loc[best_key]
F_b = news_design(EMB[emb_b].to_numpy(), row_b["h"], mode_b, 1)
fin_b = np.isfinite(F_b).all(axis=1)
by_year = {}
for Y in range(2011, dates[-1].year + 1):
    fit_m = np.asarray(dates < f"{Y}-01-01") & ok_r & fin_b
    te_y = np.asarray(dates.year == Y) & ok_r & fin_b
    if te_y.sum() < 50:
        continue
    p = fit_predict(F_b, r, fit_m, row_b["α"])[0]
    by_year[Y] = 100 * oos_r2(r, p, te_y, r[fit_m].mean())
by_year = pd.Series(by_year)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(by_year.index, by_year.values, color=np.where(by_year.values > 0, "C2", "C3"))
ax.axhline(0, color="k", lw=0.8)
ax.set_title(f"OOS R² по годам, %: {emb_b} / {mode_b} (> 0 в {(by_year > 0).mean():.0%} лет)")
plt.tight_layout(); plt.show()

---
# 6. Прогноз модуля доходности (волатильность «закрытие → закрытие»)

Знак почти непредсказуем, а величина движения — заметно лучше. Цель — $\log(|r_t|+0.01)$; база — то, что известно к
закрытию $t-1$: HAR (среднее $\log|r|$ за 1, 5 и 22 дня) и VIX на закрытии $t-1$. Новости — бак по $\varepsilon$ до
закрытия $t-1$ (форма «уровень»: волатильность — величина уровня, а не изменения). Сравниваем «база» и «база +
новости»: прирост на тесте и тест Кларка–Уэста.

В отличие от первой тетрадки, здесь в цель входит и ночной гэп, а утренние новости дня $t$ в прогноз не попадают
(они выходят после закрытия $t-1$). Поэтому эта цель шумнее и труднее, чем размах сессии.

In [ ]:
ya = M["abs_y"].to_numpy()
y1 = pd.Series(ya).shift(1)
F_base_a = np.column_stack([y1, y1.rolling(5).mean(), y1.rolling(22).mean(), M["log_vix_lag"].to_numpy()])
ok_a = np.isfinite(ya) & np.isfinite(F_base_a).all(axis=1) & (np.arange(len(dates)) >= WARMUP)
tv_a, te_a = (masks["train"] | masks["val"]) & ok_a, masks["test"] & ok_a

sb = select(lambda h: F_base_a, ya, ok_a, half_lives=[0], alphas=[1, 10, 100])
p_base_a = fit_predict(F_base_a, ya, tv_a, sb["alpha"])[0]
print(f"база (HAR + VIX) на тесте: R² = {oos_r2(ya, p_base_a, te_a, ya[tv_a].mean()):.3f}")

rows = []
for emb in EMB:
    make = lambda h, emb=emb: np.hstack([F_base_a, news_design(EMB[emb].to_numpy(), h, "уровень", 1)])
    sn = select(make, ya, ok_a, alphas=[1, 10, 100, 1000])
    F = make(sn["half_life"]); fin = np.isfinite(F).all(axis=1)
    p = fit_predict(F, ya, tv_a & fin, sn["alpha"])[0]
    m = te_a & fin
    rows.append({"эмбеддинг": emb, "h": sn["half_life"], "Δ R² val": sn["R² val"] - sb["R² val"],
                 "Δ MSE test, %": 100 * (1 - mse(ya, p, m) / mse(ya, p_base_a, m)),
                 "CW p": clark_west(ya[m], p_base_a[m], p[m])[1]})
vol_tab = pd.DataFrame(rows).set_index("эмбеддинг")
vol_tab["CW p (Холм)"] = holm(vol_tab["CW p"])
display(vol_tab.round(4))

**Как читать.** Это прямой аналог основной таблицы первой тетрадки, но для суточной доходности и с прогнозом на
закрытии. Сравните: если здесь прирост меньше, чем для размаха сессии, — значит, главную пользу давали именно утренние
новости, которые в эту схему не попадают.

---
# 7. Объяснение того же дня: новости дня $t$ → доходность дня $t$

Если сдвинуть индексы на один шаг ($\varepsilon_t$ влияет на $p_t$ сразу), получится **объясняющая** модель: она отвечает
не на вопрос «можно ли предсказать», а «какая доля движения цены за день связана с новостями того же дня». Это тоже
буквальный план, просто без задержки.

Главная опасность — **обратная причинность**: в окне дня есть статьи, вышедшие во время торгов и после закрытия,
и многие из них *пересказывают* движение («Акции упали из-за…»). Поэтому смотрим на **профиль по сдвигу**:
доходность $r_t$ сопоставляем с новостями дня $t+k$ для $k=-2,\dots,+2$ (без памяти, $h=0$):

* $k=-1, -2$ — прогноз (новости раньше доходности);
* $k=0$ — объяснение (то же окно);
* $k=+1, +2$ — **новости из будущего**. Причинного влияния здесь быть не может. Если связь есть — статьи пересказывают
  уже случившееся движение. Это мерило «пересказа».

In [ ]:
def profile(E_dict, offsets=(-2, -1, 0, 1, 2)):
    rows = []
    for emb, E in E_dict.items():
        for k in offsets:
            Ek = shift_rows(E.to_numpy(), -k)          # строка t ← новости дня t+k
            s = select(lambda h: Ek, r, ok_r, half_lives=[0])
            fin = np.isfinite(Ek).all(axis=1)
            p = fit_predict(Ek, r, TV & fin, s["alpha"])[0]
            m = TST & fin
            rows.append({"эмбеддинг": emb, "сдвиг k": k, "OOS R² test, %": 100 * oos_r2(r, p, m, ref_r),
                         "CW p": clark_west(r[m], np.full(m.sum(), ref_r), p[m])[1],
                         "IC test": spearmanr(p[m], r[m])[0] if np.std(p[m]) > 0 else np.nan})
    return pd.DataFrame(rows)

prof = profile(EMB)
display(prof.pivot(index="эмбеддинг", columns="сдвиг k", values="OOS R² test, %").round(3))

fig, ax = plt.subplots(figsize=(8, 3.4))
for emb, g_ in prof.groupby("эмбеддинг"):
    ax.plot(g_["сдвиг k"], g_["IC test"], marker="o", label=emb)
ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="0.6", ls=":", lw=1)
ax.set_xlabel("новости дня t + k"); ax.set_ylabel("IC с r_t (тест)")
ax.set_title("Профиль связи новостей и доходности по сдвигу"); ax.legend()
plt.tight_layout(); plt.show()

**Как читать профиль.** Пик на $k=0$ ожидаем: и честная реакция, и пересказ дают связь в тот же день. Решает
**асимметрия**: если связь при $k=+1$ (новости следующего дня) такая же сильная, как при $k=0$, значительная часть
«объяснения» — пересказ. Связь при $k=-1$ — это прогноз из раздела 5 (без памяти).

### 7.1. Без рыночных сводок

Уберём статьи, которые по заголовку похожи на сводки о торгах: в заголовке есть и «рынок» (stocks, shares, Wall Street,
Dow, S&P, Nasdaq, markets, futures), и глагол движения (fell, rose, gain, drop, rally, slump, surge, tumble, …).
Пересчитаем эмбеддинги без них и сравним профиль. Если связь при $k=0$ и $k=+1$ заметно падает, а при $k=-1$ — нет,
сводки отвечали за пересказ, а прогнозный сигнал от них не зависит.

In [ ]:
WRAP_MARKET = r"\b(stocks?|shares|wall st\.?|wall street|dow|s\s?&\s?p|nasdaq|markets?|futures|index)\b"
WRAP_MOVE = (r"\b(fall|falls|fell|rise|rises|rose|gain|gains|gained|drop|drops|dropped|slump|slumps|rally|rallies|"
             r"climb|climbs|tumble|tumbles|surge|surges|plunge|plunges|slide|slides|sink|sinks|soar|soars|jump|jumps|"
             r"edge|edges|retreat|retreats|rebound|rebounds|slip|slips|advance|advances|decline|declines)\b")

def is_wrap(df):
    h = df["headline"].str.lower()
    return (h.str.contains(WRAP_MARKET, regex=True) & h.str.contains(WRAP_MOVE, regex=True)).to_numpy()

wrap_masks = {g: ~is_wrap(df) for g, df in GROUPS.items()}
n_wrap = sum((~m).sum() for m in wrap_masks.values())
print(f"статей-сводок: {n_wrap:,} из {len(news):,} ({n_wrap / len(news):.1%}); примеры:")
print("\n".join("  " + t for t in news.loc[is_wrap(news), "headline"].head(8)))

EMB_nowrap = build_eps(wrap_masks)
prof_nw = profile(EMB_nowrap)
cmp = pd.concat({"все статьи": prof.pivot(index="эмбеддинг", columns="сдвиг k", values="IC test"),
                 "без сводок": prof_nw.pivot(index="эмбеддинг", columns="сдвиг k", values="IC test")}, axis=0)
display(cmp.round(3))

---
# 8. Ночь и сессия: где сидит связь

Доходность дня складывается из двух частей: $r_t = \text{гэп}_t + \text{сессия}_t$. Окно новостей дня $t$ тоже делится
на две части: статьи **до открытия** (вечер $t-1$, ночь, утро) и статьи **во время сессии**. Сопоставим каждую часть
доходности с каждой частью новостей того же дня (без памяти, $h=0$).

* **гэп ↔ новости до открытия** — самая «чистая» связь: рынок закрыт, сводок о торгах почти нет, реагирует на события
  (отчёты компаний после закрытия, ночные новости из Азии и Европы);
* **сессия ↔ новости сессии** — здесь больше всего пересказа;
* перекрёстные клетки — контроль.

И отдельно — **прогноз** обеих частей по новостям, известным к закрытию $t-1$ (форма «уровень», с подбором памяти):
какая часть доходности хоть сколько-то предсказуема.

In [ ]:
part_masks = {"до открытия": {"pre": np.ones(len(fin_pre), bool), "rest": np.zeros(len(fin_rest), bool)},
              "во время сессии": {"pre": np.zeros(len(fin_pre), bool), "rest": np.ones(len(fin_rest), bool)}}
emb_d = best_key[0]
E_parts = {name: build_eps(mm, kinds=(emb_d,))[emb_d].to_numpy() for name, mm in part_masks.items()}

rows = []
for tgt in ("gap", "sess"):
    yv = M[tgt].to_numpy()
    okv = np.isfinite(yv) & (np.arange(len(dates)) >= WARMUP)
    tv, te = (masks["train"] | masks["val"]) & okv, masks["test"] & okv
    ref = yv[tv].mean()
    row = {"часть доходности": "ночной гэп" if tgt == "gap" else "сессия"}
    for name, E in E_parts.items():
        s = select(lambda h: E, yv, okv, half_lives=[0])
        p = fit_predict(E, yv, tv, s["alpha"])[0]
        row[f"объяснение: новости {name}, IC"] = spearmanr(p[te], yv[te])[0] if np.std(p[te]) > 0 else np.nan
        row[f"объяснение: новости {name}, OOS R², %"] = 100 * oos_r2(yv, p, te, ref)
    res, _ = evaluate_signed(EMB[emb_d].to_numpy(), "уровень", lag=1, y=yv, okv=okv)
    row["прогноз (новости до закрытия t−1): OOS R², %"] = res["OOS R² test, %"]
    row["прогноз: CW p"] = res["CW p"]
    rows.append(row)
print("эмбеддинг:", emb_d)
display(pd.DataFrame(rows).set_index("часть доходности").T.round(4))

**Как читать.** Если связь сосредоточена в клетке «ночной гэп ↔ новости до открытия», новости действительно двигают
цену, и именно ночью — это согласуется с первой тетрадкой, где утренние новости оказались самыми полезными. Если
сильнее всего «сессия ↔ новости сессии», а без сводок (раздел 7.1) связь пропадает, — это в основном пересказ. Строки
«прогноз» показывают, какая часть доходности хоть сколько-то предсказуема заранее.

---
# 9. Итоги

In [ ]:
b = signed.set_index(["эмбеддинг", "форма"]).loc[best_key]
print(f"Лучшая по валидации модель знака доходности: {best_key[0]} / {best_key[1]}, h = {int(b['h'])}")
print(f"  OOS R² на тесте: {b['OOS R² test, %']:+.3f}%   CW p (Холм): {b['CW p (Холм)']:.3f}   "
      f"знаки: {b['знаки угаданы']:.1%} (PT p = {b['PT p']:.3f})")
n_sig = int((signed["CW p (Холм)"] < 0.05).sum())
print(f"  значимых моделей знака после поправки Холма: {n_sig} из {len(signed)}")
lit = signed.groupby("форма")["OOS R² test, %"].mean()
print(f"  средний OOS R² по формам: буквальная {lit.get('разность', np.nan):+.3f}%, уровень {lit.get('уровень', np.nan):+.3f}%")
print(f"  лучшая модель: OOS R² > 0 в {(by_year > 0).mean():.0%} лет")
vb = vol_tab["Δ MSE test, %"].idxmax()
print(f"Модуль доходности: лучший прирост к HAR + VIX — {vb}: {vol_tab.loc[vb, 'Δ MSE test, %']:+.2f}% "
      f"(CW p Холм = {vol_tab.loc[vb, 'CW p (Холм)']:.3f})")
k0 = prof[prof["сдвиг k"] == 0]["IC test"].mean(); k1 = prof[prof["сдвиг k"] == 1]["IC test"].mean()
km1 = prof[prof["сдвиг k"] == -1]["IC test"].mean()
print(f"Профиль по сдвигу (средний IC): прогноз k=−1: {km1:+.3f}, тот же день k=0: {k0:+.3f}, из будущего k=+1: {k1:+.3f}")

### Как читать итог

| что видим | что это значит |
|---|---|
| OOS $R^2$ знака около нуля, CW незначим после Холма | дневную доходность индекса по новостям газеты предсказать нельзя — ожидаемый и честный результат |
| значимо, но доля знаков не лучше «всегда рост» (PT незначим) | улучшение в величине, а не в направлении; для стратегии мало |
| буквальная форма лучше уровня | данные согласуются с планом: реакция на новость и её постепенный откат |
| модуль доходности предсказывается, знак — нет | новости несут информацию о неопределённости, а не о направлении |
| сильная связь при $k=0$ и почти такая же при $k=+1$ | «объяснение» того же дня в значительной мере — пересказ движения |
| связь сосредоточена в «гэп ↔ новости до открытия» | цену двигают ночные события; см. первую тетрадку, где утренние новости оказались самыми полезными |

**Связь с первой тетрадкой.** Там новости до 9:30 предсказывали размах сессии того же дня. Здесь прогноз делается на
закрытии $t-1$, и утренние новости дня $t$ в него не попадают. Если здесь сигнал слабее — это прямое подтверждение,
что полезная информация приходит в последние часы перед открытием.